# Physical Fields with Embedded Interpolants

Reproducing the four 2D physical-field benchmarks from
**Coeurdoux et al. 2026, *Training-Free Generative Modeling via Kernelized Stochastic Interpolants*** (arXiv:2602.20070, Figure 2):

1. **3D turbulence** — pressure slices from forced isotropic turbulence (JHTDB)
2. **Dark matter** — log-density slices from Quijote N-body sims
3. **Magnetic turbulence** — vorticity slices from 3D MHD (Allys et al. 2019)
4. **Weak lensing** — convergence maps (Gupta et al. 2018, Columbia Lensing)

All fields on **64 × 64** grids, periodic BC, standardised to zero-mean unit-variance.

## Strategy

The four real datasets require non-trivial download (JHTDB token, Quijote tools, FITS server, private MHD).
This notebook is designed so it **runs out-of-the-box on procedurally generated surrogate fields**
that match each target's qualitative statistics (power spectrum, anisotropy, sparsity, log-normality).
Each loader has a clear hook to swap in the real data once you have it locally.

## Pipeline

`64×64 field → flatten → PCA(d_pca) → EmbeddedInterpolants → inverse PCA → 64×64 generated`

PCA is essential here for the same reason it was essential on LFW faces: the lifting ratio η collapses
in raw pixel space (d=4096) but is healthy in PCA space at d~64–128.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from src import EmbeddedInterpolants

np.random.seed(0)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['image.interpolation'] = 'nearest'


## Surrogate field generators

Each generator produces a stack of `n` images of size `(N, N)` with the qualitative statistics of
the target dataset. Replace any of these with real loaders (see hooks below) when ready.


In [ ]:
def grf_2d(N=64, alpha=2.0, ax_aniso=1.0, n=1, rng=None):
    """
    Gaussian random field on N×N torus with isotropic-or-anisotropic power-law spectrum
    P(k) ~ k^{-alpha}.  ax_aniso != 1 stretches the spectrum along the x-axis
    (anisotropic correlation length).
    """
    rng = rng or np.random.default_rng()
    kx = np.fft.fftfreq(N)[:, None]
    ky = np.fft.fftfreq(N)[None, :]
    k = np.sqrt((ax_aniso * kx) ** 2 + (ky / ax_aniso) ** 2) + 1e-8
    spec = k ** (-alpha / 2.0)
    spec[0, 0] = 0.0   # remove DC (zero-mean field)
    out = np.empty((n, N, N), dtype=np.float32)
    for i in range(n):
        z = rng.standard_normal((N, N)) + 1j * rng.standard_normal((N, N))
        f = np.real(np.fft.ifft2(z * spec))
        f = (f - f.mean()) / (f.std() + 1e-12)
        out[i] = f.astype(np.float32)
    return out


def make_3d_turbulence(n=200, N=64):
    """Surrogate for JHTDB pressure slices: isotropic, intermittent, k^{-5/3}-ish."""
    rng = np.random.default_rng(1)
    base = grf_2d(N, alpha=5/3 * 2, ax_aniso=1.0, n=n, rng=rng)   # P(k) ~ k^{-5/3}
    # mild intermittency: multiply by independent log-normal envelope
    env = np.exp(0.4 * grf_2d(N, alpha=4.0, n=n, rng=rng))
    out = base * env
    out = (out - out.mean(axis=(1, 2), keepdims=True)) / (out.std(axis=(1, 2), keepdims=True) + 1e-12)
    return out.astype(np.float32)


def make_dark_matter(n=200, N=64):
    """Surrogate for Quijote log-density: log of an exponentiated GRF."""
    rng = np.random.default_rng(2)
    g = grf_2d(N, alpha=3.0, n=n, rng=rng)
    rho = np.exp(0.9 * g)               # log-normal density
    out = np.log(rho + 1e-3)            # the paper uses log
    out = (out - out.mean(axis=(1, 2), keepdims=True)) / (out.std(axis=(1, 2), keepdims=True) + 1e-12)
    return out.astype(np.float32)


def make_mhd_turbulence(n=200, N=64):
    """Surrogate for Allys MHD vorticity: anisotropic, signed, with elongated structures."""
    rng = np.random.default_rng(3)
    out = grf_2d(N, alpha=2.5, ax_aniso=2.0, n=n, rng=rng)   # stretched along x
    # mild non-Gaussianity by signed cube-root
    out = np.sign(out) * np.abs(out) ** 0.85
    out = (out - out.mean(axis=(1, 2), keepdims=True)) / (out.std(axis=(1, 2), keepdims=True) + 1e-12)
    return out.astype(np.float32)


def make_weak_lensing(n=200, N=64):
    """Surrogate for Gupta convergence maps: sparse positive peaks on near-Gaussian background."""
    rng = np.random.default_rng(4)
    bg = grf_2d(N, alpha=2.0, n=n, rng=rng)
    # sparse positive peaks
    peaks = np.zeros_like(bg)
    n_peaks = 25
    for i in range(n):
        ix = rng.integers(0, N, size=n_peaks)
        iy = rng.integers(0, N, size=n_peaks)
        amp = rng.exponential(scale=2.0, size=n_peaks)
        peaks[i, ix, iy] += amp
    # smooth peaks slightly
    from scipy.ndimage import gaussian_filter
    for i in range(n):
        peaks[i] = gaussian_filter(peaks[i], sigma=1.0)
    out = bg + 0.7 * peaks
    out = (out - out.mean(axis=(1, 2), keepdims=True)) / (out.std(axis=(1, 2), keepdims=True) + 1e-12)
    return out.astype(np.float32)


### Real-data hooks

When you have a real dataset locally, replace the surrogate generator with one of these patterns:

```python
# JHTDB 3D turbulence (requires pyJHTDB + token)
# Cheng et al. and Coeurdoux et al. use 2D pressure slices from `isotropic1024coarse`.
# See: https://turbulence.pha.jhu.edu

# Quijote dark matter
# Public data via globus.  Needed: 2D slices of the 3D density field at z=0.
# See: https://quijote-simulations.readthedocs.io

# Allys MHD turbulence
# Not public; contact Erwan Allys (LPENS).

# Weak lensing convergence (Gupta 2018)
# FITS files from Columbia Lensing.  See: http://columbialensing.org
# Load with astropy.io.fits, crop to 64×64, standardise.
```

The rest of the notebook only needs an array of shape `(n, 64, 64)` and a colormap, so swap and re-run.


## Generic experiment runner

Same pipeline for all four fields: PCA → EmbeddedInterpolants → inverse PCA → side-by-side grid.


In [ ]:
def run_field_experiment(
    images,            # (n, N, N) float32
    cmap='viridis',
    name='field',
    d_pca=64,
    n_iterations=8,
    n_show=5,
    K_steps=80,
    noise_level=1.5,
    q=0.5, q_final=0.15,
    gamma=1e-2, gamma_final=1e-7,
    n_inducing=100,
    seed=0,
):
    rng = np.random.default_rng(seed)
    n, N, _ = images.shape
    print(f'\n=== {name} ({n} images, {N}×{N}) ===')

    # ── train/test split ────────────────────────────────────────────────
    perm = rng.permutation(n)
    n_train = int(0.7 * n)
    train_idx, test_idx = perm[:n_train], perm[n_train:]
    X_train = images[train_idx].reshape(n_train, -1)
    X_test  = images[test_idx].reshape(n - n_train, -1)
    print(f'  train: {X_train.shape},  test: {X_test.shape}')

    # ── PCA (essential at d=4096) ──────────────────────────────────────
    pca = PCA(n_components=d_pca, whiten=False).fit(X_train)
    Z_train = pca.transform(X_train).astype(np.float32)
    print(f'  PCA: d_pca={d_pca},  explained variance={pca.explained_variance_ratio_.sum():.3f}')

    # standardise so source noise N(0, I) is well matched to target std
    sigma_z = Z_train.std(axis=0, keepdims=True) + 1e-8
    Z_train_n = Z_train / sigma_z

    # ── fit EmbeddedInterpolants ────────────────────────────────────────
    model = EmbeddedInterpolants(
        sigma_k=None, bandwidth_method='quantile',
        q=q, q_final=q_final,
        gamma=gamma, gamma_final=gamma_final,
        K_steps=K_steps, n_inducing=n_inducing,
        noise_level=noise_level, rescale=True,
    )
    src = rng.standard_normal((n_train, d_pca)).astype(np.float32)
    model.fit(src, Z_train_n, n_iterations=n_iterations, verbose=True)

    # ── generate fresh samples ─────────────────────────────────────────
    n_gen = max(n_show, 8)
    src_new = rng.standard_normal((n_gen, d_pca)).astype(np.float32)
    res = model.transport(src_new, verbose=False)
    Z_gen = res['particles'] * sigma_z              # un-standardise
    X_gen = pca.inverse_transform(Z_gen).reshape(n_gen, N, N)

    # ── visual: top row = real, bottom row = generated ──────────────────
    fig, axes = plt.subplots(2, n_show, figsize=(2 * n_show, 4.2))
    real_show = images[test_idx[:n_show]]
    vmin = min(real_show.min(), X_gen[:n_show].min())
    vmax = max(real_show.max(), X_gen[:n_show].max())
    if cmap in ('RdBu_r', 'seismic'):
        m = max(abs(vmin), abs(vmax))
        vmin, vmax = -m, m
    for j in range(n_show):
        axes[0, j].imshow(real_show[j], cmap=cmap, vmin=vmin, vmax=vmax)
        axes[1, j].imshow(X_gen[j],     cmap=cmap, vmin=vmin, vmax=vmax)
        axes[0, j].set_xticks([]); axes[0, j].set_yticks([])
        axes[1, j].set_xticks([]); axes[1, j].set_yticks([])
    axes[0, 0].set_ylabel('ground truth', fontsize=11)
    axes[1, 0].set_ylabel('generated',    fontsize=11)
    fig.suptitle(name, fontsize=13)
    plt.tight_layout()
    plt.show()

    return {'model': model, 'pca': pca, 'sigma_z': sigma_z,
            'X_gen': X_gen, 'real_show': real_show}


---
## 1. 3D turbulence (pressure)

Isotropic, multiscale, intermittent.  Diverging colormap for signed pressure fluctuations.


In [ ]:
turb_imgs = make_3d_turbulence(n=300)
res_turb = run_field_experiment(turb_imgs, cmap='RdBu_r', name='3D turbulence (pressure)',
                                d_pca=64, n_iterations=8)


---
## 2. Dark matter (log-density)

Strongly non-Gaussian, filamentary cosmic web.  `magma` for the log-density convention.


In [ ]:
dm_imgs = make_dark_matter(n=300)
res_dm = run_field_experiment(dm_imgs, cmap='magma', name='Dark matter (log-density)',
                              d_pca=80, n_iterations=10, q_final=0.10)


---
## 3. Magnetic turbulence (vorticity)

Anisotropic (mean field along x), elongated coherent structures.  Diverging colormap.


In [ ]:
mhd_imgs = make_mhd_turbulence(n=300)
res_mhd = run_field_experiment(mhd_imgs, cmap='RdBu_r', name='Magnetic turbulence (vorticity)',
                               d_pca=64, n_iterations=8)


---
## 4. Weak lensing (convergence)

Sparse positive peaks on a Gaussian-like background.  `inferno` to highlight peak structure.


In [ ]:
wl_imgs = make_weak_lensing(n=300)
res_wl = run_field_experiment(wl_imgs, cmap='inferno', name='Weak lensing (convergence)',
                              d_pca=64, n_iterations=8)


---
## 5. Combined figure (paper-style)

Single figure with all four fields side-by-side, top row real / bottom row generated.


In [ ]:
configs = [
    ('3D turbulence',      res_turb, 'RdBu_r'),
    ('Dark matter',        res_dm,   'magma'),
    ('Magnetic turbulence', res_mhd, 'RdBu_r'),
    ('Weak lensing',       res_wl,   'inferno'),
]

fig, axes = plt.subplots(2, 4, figsize=(11, 5.6))
for col, (name, r, cmap) in enumerate(configs):
    real = r['real_show'][0]
    gen  = r['X_gen'][0]
    if cmap in ('RdBu_r', 'seismic'):
        m = max(abs(real).max(), abs(gen).max())
        kw = dict(vmin=-m, vmax=m)
    else:
        vmin = min(real.min(), gen.min()); vmax = max(real.max(), gen.max())
        kw = dict(vmin=vmin, vmax=vmax)
    axes[0, col].imshow(real, cmap=cmap, **kw)
    axes[1, col].imshow(gen,  cmap=cmap, **kw)
    axes[0, col].set_title(name, fontsize=11)
    for ax in (axes[0, col], axes[1, col]):
        ax.set_xticks([]); ax.set_yticks([])

axes[0, 0].set_ylabel('ground truth', fontsize=11)
axes[1, 0].set_ylabel('generated',    fontsize=11)
plt.tight_layout()
plt.savefig('physical_fields_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: physical_fields_grid.png')


## Notes

- The bandwidth annealing schedule `q: 0.5 → 0.15` and `gamma: 1e-2 → 1e-7` is the same that worked on faces.
- For the real datasets, `d_pca` may need to grow to 100–200 to capture small-scale structure.
- `noise_level=1.5` is the default that worked on faces; for very smooth fields (3D turbulence)
  you may want to lower it to ~0.8 to avoid blurring high-frequency content.
- If you see lifting ratio η < 0.2 sustained across iterations, that's the classic signal that
  `d_pca` is too high relative to the number of training images — reduce it.
